# 05. User Affinity Features - Personalized Recommendation Signals

## Overview

**Problem:** Generic popularity features are weak (AUC 0.6777)

**Solution:** Build personalized user-song affinity features that capture:
- Does THIS user like THIS song's genre?
- Does THIS user like THIS song's artist?
- Does THIS user like songs of THIS age?
- Does THIS user use THIS platform?
- Does THIS user listen at THIS time?

**Expected Improvement:** AUC > 0.6777

## Key Insight
Deezer recommends ONE song per user. Success depends on matching user preferences to this specific recommendation. Generic "is this song popular?" is weak; personalized "does this user want this?" is strong.

## 1. Setup & Data Loading

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
RANDOM_STATE = 42

print('Libraries loaded')

Libraries loaded


### Load Preprocessed Data

In [3]:
try:
    from google.colab import drive
    #drive.mount('/content/gdrive')
    #base_path = '/content/gdrive/MyDrive/Recommender Systems (Project)/'
    drive.mount('/content/drive', force_remount=True)
    base_path= '/content/drive/MyDrive/Music Recommender System/Recommender Systems (Project)/'
    print('Using Google Colab')
except:
    base_path = './'
    print('Using local path')

df_train = pd.read_csv(base_path + 'train_processed.csv')
df_test = pd.read_csv(base_path + 'test_processed.csv')

print(f'Train: {df_train.shape}')
print(f'Test: {df_test.shape}')
print(f'\nColumns: {list(df_train.columns)}')

Mounted at /content/drive
Using Google Colab
Train: (7558184, 15)
Test: (19918, 21)

Columns: ['genre_id', 'ts_listen', 'media_id', 'album_id', 'context_type', 'release_date', 'platform_name', 'platform_family', 'media_duration', 'listen_type', 'user_gender', 'user_id', 'artist_id', 'user_age', 'is_listened']


In [4]:
# add features (for baseline models features were created after spliting to avoid leakage)


# 2. Convert timestamps
df_train['listen_datetime'] = pd.to_datetime(df_train['ts_listen'], unit='s')
df_train['listen_hour'] = df_train['listen_datetime'].dt.hour
df_train['listen_dayofweek'] = df_train['listen_datetime'].dt.dayofweek
df_train['listen_month'] = df_train['listen_datetime'].dt.month

# 3. Calculate song age
df_train['release_datetime'] = pd.to_datetime(df_train['release_date'].astype(str), format='%Y%m%d')
df_train['song_age_days'] = (df_train['listen_datetime'] - df_train['release_datetime']).dt.days

# Replace negative song_age
df_train.loc[df_train['song_age_days'] < 0, 'song_age_days'] = 755.0

# 4. Replace duration = 0
df_train.loc[df_train['media_duration'] == 0, 'media_duration'] = 222.0

# 5. Add popularity features
user_listen_count = df_train.groupby('user_id').size().reset_index(name='user_listen_count')
item_listen_count = df_train.groupby("media_id").size().reset_index(name="item_listen_count")
artist_listen_count = df_train.groupby("artist_id").size().reset_index(name="artist_listen_count")
genre_listen_count = df_train.groupby("genre_id").size().reset_index(name="genre_listen_count")

df_train = df_train.merge(user_listen_count, on='user_id', how='left')
df_train = df_train.merge(item_listen_count, on='media_id', how='left')
df_train = df_train.merge(artist_listen_count, on='artist_id', how='left')
df_train = df_train.merge(genre_listen_count, on='genre_id', how='left')

# 6. Drop original timestamp columns
df_train = df_train.drop(columns=['ts_listen', 'listen_datetime', 'release_date', 'release_datetime'])

print(f"Test processed shape: {df_train.shape}")
print(f"Test columns: {df_train.columns.tolist()}")

Test processed shape: (7558184, 21)
Test columns: ['genre_id', 'media_id', 'album_id', 'context_type', 'platform_name', 'platform_family', 'media_duration', 'listen_type', 'user_gender', 'user_id', 'artist_id', 'user_age', 'is_listened', 'listen_hour', 'listen_dayofweek', 'listen_month', 'song_age_days', 'user_listen_count', 'item_listen_count', 'artist_listen_count', 'genre_listen_count']


## 2. Phase 1: Build User Preference Profiles

**Goal:** For each user, extract their historical preferences

In [5]:
print('='*70)
print('PHASE 1: Building User Preference Profiles')
print('='*70)

# Initialize storage
user_profiles = {}

# Get unique users
unique_users = df_train['user_id'].unique()
print(f'\nProcessing {len(unique_users):,} unique users...')

# Build profile for each user
for i, user_id in enumerate(unique_users):
    if (i + 1) % 5000 == 0:
        print(f'  Progress: {i+1:,}/{len(unique_users):,}')

    user_data = df_train[df_train['user_id'] == user_id]

    # Genre distribution: what % of user's listens per genre?
    genre_dist = user_data['genre_id'].value_counts()
    genre_dist_normalized = (genre_dist / genre_dist.sum()).to_dict()

    # Artist distribution: what % of user's listens per artist?
    artist_dist = user_data['artist_id'].value_counts()
    artist_dist_normalized = (artist_dist / artist_dist.sum()).to_dict()

    # Song age statistics
    song_ages = user_data['song_age_days'].values

    # Platform distribution
    platform_dist = user_data['platform_name'].value_counts()
    platform_dist_normalized = (platform_dist / platform_dist.sum()).to_dict()

    # Hour distribution (when does user listen?)
    hour_dist = user_data['listen_hour'].value_counts()
    hour_dist_normalized = (hour_dist / hour_dist.sum()).to_dict()

    # Store profile
    user_profiles[user_id] = {
        'genre_dist': genre_dist_normalized,
        'artist_dist': artist_dist_normalized,
        'song_age_mean': song_ages.mean(),
        'song_age_std': max(song_ages.std(), 1),  # Avoid division by zero
        'platform_dist': platform_dist_normalized,
        'hour_dist': hour_dist_normalized,
        'total_listens': len(user_data),
    }

print(f'\n✓ Built profiles for {len(user_profiles):,} users')

PHASE 1: Building User Preference Profiles

Processing 19,918 unique users...
  Progress: 5,000/19,918
  Progress: 10,000/19,918
  Progress: 15,000/19,918

✓ Built profiles for 19,918 users


### Example User Profile

In [6]:
# Show example profile
example_user = list(user_profiles.keys())[0]
profile = user_profiles[example_user]

print(f'Example User Profile (user_id={example_user})')
print(f'\nTotal listens: {profile["total_listens"]}')
print(f'\nTop 5 genres (by % of listens):')
for genre, pct in sorted(profile['genre_dist'].items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f'  Genre {genre}: {pct*100:.1f}%')
print(f'\nAvg song age: {profile["song_age_mean"]:.0f} days')
print(f'Peak listening hour: {max(profile["hour_dist"], key=profile["hour_dist"].get)} (24-hour format)')

Example User Profile (user_id=9241)

Total listens: 226

Top 5 genres (by % of listens):
  Genre 0: 26.5%
  Genre 7: 17.7%
  Genre 25: 17.3%
  Genre 2744: 5.8%
  Genre 10: 5.3%

Avg song age: 2880 days
Peak listening hour: 12 (24-hour format)


## 3. Phase 2: Extract Affinity Features for Training Data

**Goal:** For each (user, song) pair, calculate how well they match

In [7]:
print('='*70)
print('PHASE 2: Extracting User-Song Affinity Features')
print('='*70)

def extract_affinity_features(df, user_profiles, name=''):
    """
    Extract affinity features for each (user, song) pair.

    Features:
    1. genre_affinity: % of user's listens in this song's genre
    2. artist_affinity: % of user's listens to this artist
    3. age_compatibility: how close is song age to user's preference?
    4. platform_match: % of user's listens on this platform
    5. time_match: % of user's listens at this hour
    """

    n = len(df)
    features = pd.DataFrame(index=df.index)

    print(f'Processing {n:,} rows {name}...')

    # Initialize feature arrays
    genre_aff = np.zeros(n)
    artist_aff = np.zeros(n)
    age_compat = np.zeros(n)
    platform_match = np.zeros(n)
    time_match = np.zeros(n)

    # Extract features row by row
    for idx, row in df.iterrows():
        if (idx + 1) % 500000 == 0:
            print(f'  Progress: {idx+1:,}/{n:,}')

        user_id = row['user_id']
        song_genre = row['genre_id']
        song_artist = row['artist_id']
        song_age = row['song_age_days']
        platform = row['platform_name']
        hour = row['listen_hour']

        # Get user profile (or use defaults for cold-start users)
        if user_id in user_profiles:
            profile = user_profiles[user_id]
        else:
            # Cold-start user: use global averages
            profile = {
                'genre_dist': {},
                'artist_dist': {},
                'song_age_mean': df_train['song_age_days'].mean(),
                'song_age_std': max(df_train['song_age_days'].std(), 1),
                'platform_dist': {},
                'hour_dist': {},
            }

        # Feature 1: Genre Affinity
        # "What % of this user's listens were in this genre?"
        genre_aff[idx] = profile['genre_dist'].get(song_genre, 0.0)

        # Feature 2: Artist Affinity
        # "What % of this user's listens were this artist?"
        artist_aff[idx] = profile['artist_dist'].get(song_artist, 0.0)

        # Feature 3: Age Compatibility
        # "How close is this song's age to user's average?"
        age_diff = abs(song_age - profile['song_age_mean'])
        age_compat[idx] = 1.0 / (1.0 + (age_diff / max(profile['song_age_std'], 1)))

        # Feature 4: Platform Match
        # "What % of this user's listens were on this platform?"
        platform_match[idx] = profile['platform_dist'].get(platform, 0.0)

        # Feature 5: Time Match
        # "What % of this user's listens were at this hour?"
        time_match[idx] = profile['hour_dist'].get(hour, 0.0)

    features['genre_affinity'] = genre_aff
    features['artist_affinity'] = artist_aff
    features['age_compatibility'] = age_compat
    features['platform_match'] = platform_match
    features['time_match'] = time_match

    return features

# Extract features for training data
train_affinity = extract_affinity_features(df_train, user_profiles, '(train)')
test_affinity = extract_affinity_features(df_test, user_profiles, 'test')
print(f'✓ Extracted affinity features for training data')
print(f'Shape: {train_affinity.shape}')
print(f'✓ Extracted affinity features for test data')
print(f'Shape: {test_affinity.shape}')

PHASE 2: Extracting User-Song Affinity Features
Processing 7,558,184 rows (train)...
  Progress: 500,000/7,558,184
  Progress: 1,000,000/7,558,184
  Progress: 1,500,000/7,558,184
  Progress: 2,000,000/7,558,184
  Progress: 2,500,000/7,558,184
  Progress: 3,000,000/7,558,184
  Progress: 3,500,000/7,558,184
  Progress: 4,000,000/7,558,184
  Progress: 4,500,000/7,558,184
  Progress: 5,000,000/7,558,184
  Progress: 5,500,000/7,558,184
  Progress: 6,000,000/7,558,184
  Progress: 6,500,000/7,558,184
  Progress: 7,000,000/7,558,184
  Progress: 7,500,000/7,558,184
Processing 19,918 rows test...
✓ Extracted affinity features for training data
Shape: (7558184, 5)
✓ Extracted affinity features for test data
Shape: (19918, 5)


### Feature Statistics

In [8]:
print('\nAffinity Features - Statistical Summary')
print('='*70)
print(train_affinity.describe())

print('\n\nFeature Correlations with is_listened')
print('='*70)
for col in train_affinity.columns:
    corr, pval = spearmanr(train_affinity[col], df_train['is_listened'])
    print(f'{col:25s}: corr={corr:+.4f}, pval={pval:.2e}')


Affinity Features - Statistical Summary
       genre_affinity  artist_affinity  age_compatibility  platform_match  \
count    7.558184e+06     7.558184e+06       7.558184e+06    7.558184e+06   
mean     3.443578e-01     5.080989e-02       6.641641e-01    9.005756e-01   
std      3.105489e-01     1.152301e-01       1.668837e-01    2.019621e-01   
min      1.583030e-04     1.583030e-04       3.264979e-02    2.398082e-04   
25%      5.314534e-02     3.508772e-03       5.700310e-01    9.216216e-01   
50%      2.548701e-01     1.037037e-02       6.658581e-01    1.000000e+00   
75%      6.025180e-01     3.846154e-02       7.759677e-01    1.000000e+00   
max      1.000000e+00     1.000000e+00       1.000000e+00    1.000000e+00   

         time_match  
count  7.558184e+06  
mean   1.169522e-01  
std    1.126232e-01  
min    2.254791e-04  
25%    5.406488e-02  
50%    8.672087e-02  
75%    1.378882e-01  
max    1.000000e+00  


Feature Correlations with is_listened
genre_affinity           : 

### Add Further Features: Recent Preferences (last 30 days)

In [9]:
def extract_affinity_features_v2(df, user_profiles, name=''):
    """
    IMPROVED: Add recent preferences (last 30 days)
    """
    n = len(df)
    features = pd.DataFrame(index=df.index)

    genre_aff = np.zeros(n)
    artist_aff = np.zeros(n)

    # NEW: Recent preferences
    recent_genre_aff = np.zeros(n)
    recent_artist_aff = np.zeros(n)

    for idx, row in df.iterrows():
        if (idx + 1) % 500000 == 0:
            print(f'  Progress: {idx+1:,}/{n:,}')

        user_id = row['user_id']
        song_genre = row['genre_id']
        song_artist = row['artist_id']

        if user_id in user_profiles:
            profile = user_profiles[user_id]
        else:
            profile = {
                'genre_dist': {},
                'artist_dist': {},
                'recent_genre_dist': {},  # NEW
                'recent_artist_dist': {},  # NEW
            }

        # All-time affinity
        genre_aff[idx] = profile['genre_dist'].get(song_genre, 0.0)
        artist_aff[idx] = profile['artist_dist'].get(song_artist, 0.0)

        # RECENT affinity (what user liked in last 30 days)
        recent_genre_aff[idx] = profile.get('recent_genre_dist', {}).get(song_genre, 0.0)
        recent_artist_aff[idx] = profile.get('recent_artist_dist', {}).get(song_artist, 0.0)

    features['genre_affinity'] = genre_aff
    features['artist_affinity'] = artist_aff
    features['recent_genre_affinity'] = recent_genre_aff  # NEW
    features['recent_artist_affinity'] = recent_artist_aff  # NEW

    return features

### Add Interaction Features

In [10]:
def add_interaction_features(df, user_profiles):
    """
    Add interaction features based on user profile concentration

    Args:
        df: DataFrame with user_id column
        user_profiles: Dictionary of user profiles

    Returns:
        DataFrame with interaction features
    """

    print(f"Processing {len(df):,} rows for interaction features...")

    features = pd.DataFrame(index=df.index)

    # Feature 1: User Genre Focus (concentration in top genres)
    user_genre_focus = []
    for idx, user_id in enumerate(df['user_id']):
        if (idx + 1) % 500000 == 0:
            print(f"  Progress: {idx+1:,}/{len(df):,}")

        if user_id in user_profiles:
            profile = user_profiles[user_id]
            genres = list(profile['genre_dist'].values())

            if len(genres) > 0:
                # Herfindahl index: concentration measure
                # Higher = more concentrated on few genres
                hhi = sum(p**2 for p in genres)
            else:
                hhi = 0
        else:
            hhi = 0

        user_genre_focus.append(hhi)

    # Feature 2: User Artist Focus (concentration in favorite artists)
    user_artist_focus = []
    for idx, user_id in enumerate(df['user_id']):
        if (idx + 1) % 500000 == 0:
            print(f"  Progress: {idx+1:,}/{len(df):,}")

        if user_id in user_profiles:
            profile = user_profiles[user_id]
            artists = list(profile['artist_dist'].values())

            if len(artists) > 0:
                hhi = sum(p**2 for p in artists)
            else:
                hhi = 0
        else:
            hhi = 0

        user_artist_focus.append(hhi)

    # Feature 3: User Activity (total listens, normalized)
    user_activity = []
    for user_id in df['user_id']:
        if user_id in user_profiles:
            total_listens = user_profiles[user_id]['total_listens']
        else:
            total_listens = 0
        user_activity.append(total_listens)

    features['user_genre_focus'] = user_genre_focus
    features['user_artist_focus'] = user_artist_focus
    features['user_total_listens'] = user_activity

    return features

In [11]:
# ============================================================================
# NEW CELL: EXTRACT NEW AFFINITY FEATURES (Corrected)
# ============================================================================

print("\n" + "="*80)
print("EXTRACTING NEW AFFINITY FEATURES (Recent + Interaction)")
print("="*80)

# ============ OPTION 1: EXTRACT RECENT AFFINITY FEATURES ============
print("\n[Phase 1] Extracting RECENT AFFINITY FEATURES...")
print("  (This may take a few minutes)")

train_recent_affinity = extract_affinity_features_v2(df_train, user_profiles, '(train - recent)')
test_recent_affinity = extract_affinity_features_v2(df_test, user_profiles, '(test - recent)')

print(f"  ✓ train_recent_affinity: {train_recent_affinity.shape}")
print(f"  ✓ test_recent_affinity: {test_recent_affinity.shape}")
print(f"\n  Columns: {list(train_recent_affinity.columns)}")

# ============ OPTION 2: EXTRACT INTERACTION FEATURES ============
print("\n[Phase 2] Extracting INTERACTION FEATURES...")

train_interaction_features = add_interaction_features(df_train, user_profiles)
test_interaction_features = add_interaction_features(df_test, user_profiles)

print(f"  ✓ train_interaction_features: {train_interaction_features.shape}")
print(f"  ✓ test_interaction_features: {test_interaction_features.shape}")
print(f"\n  Columns: {list(train_interaction_features.columns)}")

# ============ COMBINE ALL AFFINITY FEATURES ============
print("\n[Phase 3] Combining ALL affinity features...")

train_affinity_all = pd.concat([
    train_affinity.reset_index(drop=True),
    train_recent_affinity.reset_index(drop=True),
    train_interaction_features.reset_index(drop=True)
], axis=1)

test_affinity_all = pd.concat([
    test_affinity.reset_index(drop=True),
    test_recent_affinity.reset_index(drop=True),
    test_interaction_features.reset_index(drop=True)
], axis=1)

print(f"\n  ✓ train_affinity_all: {train_affinity_all.shape}")
print(f"  ✓ test_affinity_all: {test_affinity_all.shape}")

# ============ SUMMARY ============
print("\n" + "="*80)
print("FEATURE SUMMARY")
print("="*80)
print(f"\nTotal affinity features: {len(train_affinity_all.columns)}")
print(f"Breakdown:")
print(f"  - Original affinity: {len(train_affinity.columns)}")
print(f"  - Recent affinity: {len(train_recent_affinity.columns)}")
print(f"  - Interaction features: {len(train_interaction_features.columns)}")
print(f"\nAll columns:")
print(list(train_affinity_all.columns))

print("\n✓ Ready to integrate into training data!")
print("="*80 + "\n")


EXTRACTING NEW AFFINITY FEATURES (Recent + Interaction)

[Phase 1] Extracting RECENT AFFINITY FEATURES...
  (This may take a few minutes)
  Progress: 500,000/7,558,184
  Progress: 1,000,000/7,558,184
  Progress: 1,500,000/7,558,184
  Progress: 2,000,000/7,558,184
  Progress: 2,500,000/7,558,184
  Progress: 3,000,000/7,558,184
  Progress: 3,500,000/7,558,184
  Progress: 4,000,000/7,558,184
  Progress: 4,500,000/7,558,184
  Progress: 5,000,000/7,558,184
  Progress: 5,500,000/7,558,184
  Progress: 6,000,000/7,558,184
  Progress: 6,500,000/7,558,184
  Progress: 7,000,000/7,558,184
  Progress: 7,500,000/7,558,184
  ✓ train_recent_affinity: (7558184, 4)
  ✓ test_recent_affinity: (19918, 4)

  Columns: ['genre_affinity', 'artist_affinity', 'recent_genre_affinity', 'recent_artist_affinity']

[Phase 2] Extracting INTERACTION FEATURES...
Processing 7,558,184 rows for interaction features...
  Progress: 500,000/7,558,184
  Progress: 1,000,000/7,558,184
  Progress: 1,500,000/7,558,184
  Progress:

## 4. Test Set Feature Extraction

In [12]:
# ============ COMBINE ALL AFFINITY FEATURES ============
print("\n[Phase 3] Combining ALL affinity features...")

train_affinity_all = pd.concat([
    train_affinity.reset_index(drop=True),
    train_recent_affinity.reset_index(drop=True),
    train_interaction_features.reset_index(drop=True)
], axis=1)

test_affinity_all = pd.concat([
    test_affinity.reset_index(drop=True),
    test_recent_affinity.reset_index(drop=True),
    test_interaction_features.reset_index(drop=True)
], axis=1)

print(f"\n  ✓ train_affinity_all: {train_affinity_all.shape}")
print(f"  ✓ test_affinity_all: {test_affinity_all.shape}")


[Phase 3] Combining ALL affinity features...

  ✓ train_affinity_all: (7558184, 12)
  ✓ test_affinity_all: (19918, 12)


## 5. Prepare Data for Modeling

In [13]:
from re import X

print("\n" + "="*80)
print("CELL 17: LOAD DATA + ADD AFFINITY FEATURES")
print("="*80)

# ============ 1. LOAD CHRONOLOGICALLY SPLIT DATA ============
print("\n[Step 1] Loading chronologically split data...")

train_data = pd.read_csv(base_path + 'train_data.csv')
val_data = pd.read_csv(base_path + 'val_data.csv')
test_data = pd.read_csv(base_path + 'test_data.csv')

print(f"  train_data: {train_data.shape}")
print(f"  val_data: {val_data.shape}")
print(f"  test_data: {test_data.shape}")

# ============ 2. DEFINE ORIGINAL FEATURES ============
print("\n[Step 2] Defining original features (16 columns)...")

target_col = 'is_listened'

feature_cols = [
    'genre_id', 'context_type', 'platform_name', 'platform_family',
    'listen_type', 'user_gender',
    'listen_hour', 'listen_dayofweek', 'listen_month',
    'media_duration', 'user_age', 'song_age_days',
    'user_listen_count', 'item_listen_count',
    'artist_listen_count', 'genre_listen_count'
]

print(f"  Original features: {len(feature_cols)}")
print(f"  {feature_cols}")

# ============ 3. CREATE ORIGINAL FEATURE MATRICES ============
print("\n[Step 3] Creating original feature matrices...")

x_train = train_data[feature_cols].copy()
y_train = train_data[target_col].copy()

x_val = val_data[feature_cols].copy()
y_val = val_data[target_col].copy()

x_test = test_data[feature_cols].copy()
y_test = test_data[target_col].copy()

print(f"  x_train: {x_train.shape}")
print(f"  x_val: {x_val.shape}")
print(f"  x_test: {x_test.shape}")

# ============ 4. HANDLE ALL AFFINITY FEATURES ============
print("\n[Step 4] Reusing all extracted affinity features...")

# Match by position (chronological order)
train_affinity_subset = train_affinity_all.iloc[:len(train_data)].reset_index(drop=True)
val_affinity_subset = train_affinity_all.iloc[len(train_data):len(train_data)+len(val_data)].reset_index(drop=True)
test_affinity_subset = test_affinity_all.iloc[:len(test_data)].reset_index(drop=True)

print(f"  ✓ train_affinity_subset: {train_affinity_subset.shape}")
print(f"  ✓ val_affinity_subset: {val_affinity_subset.shape}")
print(f"  ✓ test_affinity_subset: {test_affinity_subset.shape}")

# ============ 5. CONCATENATE ALL AFFINITY FEATURES ============
print("\n[Step 5] Concatenating all affinity features...")

x_train = pd.concat([x_train.reset_index(drop=True), train_affinity_subset], axis=1)
x_val = pd.concat([x_val.reset_index(drop=True), val_affinity_subset], axis=1)
x_test = pd.concat([x_test.reset_index(drop=True), test_affinity_subset], axis=1)

print(f"  x_train WITH all affinity: {x_train.shape}")
print(f"  x_val WITH all affinity: {x_val.shape}")
print(f"  x_test WITH all affinity: {x_test.shape}")

# ============ 6. VERIFY ALL COLUMNS ============
print(f"\n[Step 6] Total features: {len(x_train.columns)}")
print(f"  Original: 16")
print(f"  All affinity: {len(x_train.columns) - 16}")
print(list(x_train.columns))
affinity_cols = [c for c in x_train.columns if c not in feature_cols]

# ============ 7. SUMMARY ============
print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"Original features: {len(feature_cols)}")
print(f"Affinity features: {len(affinity_cols)}")
print(f"Total features: {len(x_train.columns)}")
print(f"\nFeature columns:")
print(list(x_train.columns))
print(f"\nx_train shape: {x_train.shape}")
print(f"x_val shape: {x_val.shape}")
print(f"x_test shape: {x_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_val shape: {y_val.shape}")
print(f"y_test shape: {y_test.shape}")
print("\n✓ Ready for model training with affinity features!")
print("="*80 + "\n")


CELL 17: LOAD DATA + ADD AFFINITY FEATURES

[Step 1] Loading chronologically split data...
  train_data: (6046547, 21)
  val_data: (755818, 21)
  test_data: (755819, 21)

[Step 2] Defining original features (16 columns)...
  Original features: 16
  ['genre_id', 'context_type', 'platform_name', 'platform_family', 'listen_type', 'user_gender', 'listen_hour', 'listen_dayofweek', 'listen_month', 'media_duration', 'user_age', 'song_age_days', 'user_listen_count', 'item_listen_count', 'artist_listen_count', 'genre_listen_count']

[Step 3] Creating original feature matrices...
  x_train: (6046547, 16)
  x_val: (755818, 16)
  x_test: (755819, 16)

[Step 4] Reusing all extracted affinity features...
  ✓ train_affinity_subset: (6046547, 12)
  ✓ val_affinity_subset: (755818, 12)
  ✓ test_affinity_subset: (19918, 12)

[Step 5] Concatenating all affinity features...
  x_train WITH all affinity: (6046547, 28)
  x_val WITH all affinity: (755818, 28)
  x_test WITH all affinity: (755819, 28)

[Step 6]

In [14]:
# ============================================================================
# FIX: Remove duplicate columns
# ============================================================================

print("\n" + "="*80)
print("REMOVING DUPLICATE COLUMNS")
print("="*80)

print(f"\nBefore: {len(x_train.columns)} columns")
print(f"Columns: {list(x_train.columns)}")

# Find and drop duplicates (keep first occurrence)
x_train_clean = x_train.loc[:, ~x_train.columns.duplicated(keep='first')]
x_val_clean = x_val.loc[:, ~x_val.columns.duplicated(keep='first')]
x_test_clean = x_test.loc[:, ~x_test.columns.duplicated(keep='first')]

# Replace with clean versions
x_train = x_train_clean
x_val = x_val_clean
x_test = x_test_clean

print(f"\nAfter: {len(x_train.columns)} columns")
print(f"Columns: {list(x_train.columns)}")

print(f"\n✓ Duplicates removed!")
print("="*80 + "\n")


REMOVING DUPLICATE COLUMNS

Before: 28 columns
Columns: ['genre_id', 'context_type', 'platform_name', 'platform_family', 'listen_type', 'user_gender', 'listen_hour', 'listen_dayofweek', 'listen_month', 'media_duration', 'user_age', 'song_age_days', 'user_listen_count', 'item_listen_count', 'artist_listen_count', 'genre_listen_count', 'genre_affinity', 'artist_affinity', 'age_compatibility', 'platform_match', 'time_match', 'genre_affinity', 'artist_affinity', 'recent_genre_affinity', 'recent_artist_affinity', 'user_genre_focus', 'user_artist_focus', 'user_total_listens']

After: 26 columns
Columns: ['genre_id', 'context_type', 'platform_name', 'platform_family', 'listen_type', 'user_gender', 'listen_hour', 'listen_dayofweek', 'listen_month', 'media_duration', 'user_age', 'song_age_days', 'user_listen_count', 'item_listen_count', 'artist_listen_count', 'genre_listen_count', 'genre_affinity', 'artist_affinity', 'age_compatibility', 'platform_match', 'time_match', 'recent_genre_affinity',

In [15]:
key_cols = [
  'user_id','media_id','album_id','artist_id','genre_id',
  'context_type','platform_name','platform_family','listen_type',
  'user_gender','user_age','listen_hour','listen_dayofweek','listen_month',
  'song_age_days','user_listen_count','item_listen_count','artist_listen_count','genre_listen_count'
]

train_ids = train_data[key_cols].reset_index(drop=True)
val_ids   = val_data[key_cols].reset_index(drop=True)
aff_train_ids = df_train[key_cols].reset_index(drop=True)

train_align = (train_ids == aff_train_ids.iloc[:len(train_ids)].values).all(axis=1).mean()
val_align   = (val_ids == aff_train_ids.iloc[len(train_ids):len(train_ids)+len(val_ids)].values).all(axis=1).mean()

print("Alignment rate (train):", train_align)
print("Alignment rate (val):  ", val_align)

Alignment rate (train): 0.0
Alignment rate (val):   0.0


In [17]:




N = 5000  # keep small
cols = ['user_id', 'media_id', 'listen_hour', 'listen_dayofweek', 'listen_month']

a = train_data[cols].dropna().sample(min(N, len(train_data)), random_state=42)
b = val_data[cols].dropna().sample(min(N, len(val_data)), random_state=42)

ah = pd.util.hash_pandas_object(a.fillna("<NA>"), index=False)
bh = pd.util.hash_pandas_object(b.fillna("<NA>"), index=False)

overlap = int(pd.Index(ah).isin(pd.Index(bh)).sum())
print("Sample overlap (richer fingerprint):", overlap)

Sample overlap (richer fingerprint): 0


## 6. Model Validation: LightGBM with Affinity Features

In [ ]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

print('='*70)
print('MODEL: LightGBM with Affinity Features')
print('='*70)

# Train model
lgb_affinity = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.05,
    num_leaves=31,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum()
)

lgb_affinity.fit(
    x_train, y_train,
    eval_set=[(x_val, y_val)],
    callbacks=[lgb.log_evaluation(period=0)]
)

# Predict
y_pred_affinity = lgb_affinity.predict_proba(x_test)[:, 1]
auc_affinity = roc_auc_score(y_test, y_pred_affinity)

print(f'\nTest AUC (Affinity Features): {auc_affinity:.4f}')

# Feature importance
importance_df = pd.DataFrame({
    'feature': x_train.columns,
    'importance': lgb_affinity.feature_importances_
}).sort_values('importance', ascending=False)

print(f'\nFeature Importance:')
print(importance_df.to_string(index=False))

## 7. Comparison to Baseline

In [ ]:
baseline_auc = 0.6777  # From 03_baseline_model -> LightGBM
improvement = auc_affinity - baseline_auc
improvement_pct = (improvement / baseline_auc) * 100

print('\n' + '='*70)
print('RESULTS COMPARISON')
print('='*70)
print(f'\nBaseline AUC (generic features):  {baseline_auc:.4f}')
print(f'Affinity AUC (personalized):      {auc_affinity:.4f}')
print(f'\nImprovement:                     +{improvement:.4f} ({improvement_pct:+.1f}%)')

if auc_affinity > baseline_auc:
    print(f'\n✓ SUCCESS: Affinity features improved AUC!')
else:
    print(f'\n⚠️ No improvement. Check feature calculation.')

## 8. Visualizations

In [ ]:
# Feature importance plot
fig, ax = plt.subplots(figsize=(10, 6))
importance_df.plot(x='feature', y='importance', kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Importance')
ax.set_title('User Affinity Feature Importance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Feature correlation heatmap
fig, ax = plt.subplots(figsize=(8, 6))
corr_matrix = train_affinity.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax, vmin=-1, vmax=1)
ax.set_title('Affinity Features Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Distribution Analysis

In [ ]:
# Show distribution of each feature by is_listened
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Feature Distributions by is_listened', fontsize=14, fontweight='bold')

for idx, col in enumerate(train_affinity.columns):
    ax = axes[idx // 3, idx % 3]

    # Plot distributions
    listened = train_affinity[df_train['is_listened'] == 1][col]
    not_listened = train_affinity[df_train['is_listened'] == 0][col]

    ax.hist(not_listened, alpha=0.6, label='Not Listened', bins=30, color='red')
    ax.hist(listened, alpha=0.6, label='Listened', bins=30, color='green')
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.set_yscale('log')

# Remove extra subplot
fig.delaxes(axes[1, 2])
plt.tight_layout()
plt.show()

## 10. Save Features for Next Phase

In [ ]:
# Save for use in next notebooks (ensemble, embeddings, etc.)
try:
    train_affinity.to_csv(base_path + 'train_affinity_features.csv', index=False)
    test_affinity.to_csv(base_path + 'test_affinity_features.csv', index=False)
    print(f'✓ Saved affinity features')
except:
    print('Could not save (maybe no write permission)')

# Also save user_profiles for later use
import pickle
try:
    pickle.dump(user_profiles, open(base_path + 'user_profiles.pkl', 'wb'))
    print(f'✓ Saved user profiles')
except:
    print('Could not save user profiles')

## 11. Summary & Next Steps

In [ ]:
print('\n' + '='*70)
print('SUMMARY: USER AFFINITY FEATURES')
print('='*70)

print(f'''
✓ COMPLETED:
  1. Built user preference profiles from {len(user_profiles):,} users
  2. Extracted 5 affinity features:
     - Genre affinity (user's preference for this song's genre)
     - Artist affinity (user's familiarity with artist)
     - Age compatibility (match of song age to user's taste)
     - Platform match (user's platform preference)
     - Time match (user's listening hour pattern)
  3. Validated features correlate with is_listened
  4. Trained LightGBM model

📊 RESULTS:
  Baseline AUC (generic features):     {baseline_auc:.4f}
  Affinity AUC (personalized):         {auc_affinity:.4f}
  Improvement:                         +{improvement:.4f} ({improvement_pct:+.1f}%)

🔑 KEY INSIGHT:
  Personalization matters! User affinity features capture the essence of
  "Does THIS user want THIS song?" which is what Deezer cares about.

📋 NEXT STEPS:
  1. Combine with other features (original + embeddings) in 06_collaborative_embeddings.ipynb
  2. Build ensemble models
  3. Optimize hyperparameters
  4. Target: Get AUC to 0.66+

✅ DELIVERABLE:
  - train_affinity_features.csv
  - test_affinity_features.csv
  - user_profiles.pkl (for reproducibility)
  - This notebook (full documentation)
''')